In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import random
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold, cross_val_predict, cross_validate
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, make_scorer

### Check the performance of Random Forests with different training samples

In [ ]:
# Set seed for reproducibility and test_size = 0.2
seed = 25
np.random.seed(seed)
random.seed(seed)

# Number of Fold in cross validation
n_folds = 5   

# Read the updated SOC dataset (baseline 904 + new 100 SOC observations)
combined_soc_data = (r"updated_training_data.gpkg")

# Use optimal hyperparameters found by 5-fold cross-validation for the baseline RF model
best_params = {'bootstrap': False, 'criterion': 'squared_error', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 100}

#### Load the data

In [ ]:
data = gpd.read_file(combined_soc_data)
print("rows and columns:", data.shape)
print("columns:", list(data.columns))

# Which column says whether a sample is old or new?  Print it and check the labels.
source_column = "origin"          # confirmed from the file
print("\nlabels in the source column:")
print(data[source_column].value_counts())

# True for the 100 new samples, False for the 904 legacy samples.
is_new = (data[source_column] == "field_work").values # this variable is used to remove any fieldwork samples during the cross validation for baseline RF model
print("\nnew samples found:", is_new.sum(), " legacy samples:", (~is_new).sum())

#### Preprocessing the dataset

In [ ]:
# Separate the dataset into X and y
# Doing get_dummies once (not separately per model) guarantees both models
# see exactly the same feature columns.
X = data.drop(columns=["soc", source_column, "geometry"])
X = pd.get_dummies(X)
y = data["soc"].values

print("\nfeatures used:", X.shape[1])
print("Number of data:", X.shape[0])
print(list(X.columns))

#### Random Cross-Validation

In [ ]:
# Every sample gets a fold number from 0 to 4.
fold_number = np.zeros(len(data), dtype = int)

kf = KFold(n_splits = n_folds, shuffle = True, random_state = seed)
for i, (train_rows, test_rows) in enumerate(kf.split(X)):
    fold_number[test_rows] = i # Assign the fold number to the testing set for each loop

print("\nsamples per fold:", np.bincount(fold_number))
print("new samples per fold:", [int(is_new[fold_number == i].sum()) for i in range(n_folds)])

In [ ]:
# Create the paired loop
# Two empty arrays to collect one prediction per sample.
pred_baseline = np.zeros(len(data))
pred_updated = np.zeros(len(data))

for i in range(n_folds):
    is_test = (fold_number == i)          # this fold is the test set
    is_train = ~is_test                   # the other 4 folds are the training set

    # Updated RF model: trains on everything in the training folds
    model_updated = RandomForestRegressor(**best_params, random_state = seed)
    model_updated.fit(X[is_train], y[is_train])
    pred_updated[is_test] = model_updated.predict(X[is_test])

    # Baseline model: same training folds, but the new samples are removed
    is_train_baseline = is_train & (~is_new)
    model_baseline = RandomForestRegressor(**best_params, random_state = seed)
    model_baseline.fit(X[is_train_baseline], y[is_train_baseline])
    pred_baseline[is_test] = model_baseline.predict(X[is_test])

    # Full accounting, so you can see where every sample went.
    # The test fold holds both legacy and new samples, which is why the baseline training count is far below 904: the legacy samples sitting in the test fold are held out too.
    legacy_train = int((is_train & ~is_new).sum())
    legacy_test = int((is_test & ~is_new).sum())
    new_train = int((is_train & is_new).sum())
    new_test = int((is_test & is_new).sum())
 
    print(f"fold {i}: test={is_test.sum():4d}  updated_train={is_train.sum():4d}  "
          f"baseline_train={is_train_baseline.sum():4d}  ||  "
          f"legacy {legacy_train}+{legacy_test}={legacy_train + legacy_test}   "
          f"new {new_train}+{new_test}={new_train + new_test}")
 
    assert legacy_train + legacy_test == 904
    assert new_train + new_test == 100
    assert is_train_baseline.sum() == legacy_train

#### Report performance metrics

In [ ]:
def score(y_true, y_pred):
    """R2, RMSE, MAE and mean error for one set of predictions."""
    return {
        "n": len(y_true),
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "ME": np.mean(y_pred - y_true),         
    }

results = pd.DataFrame([
    score(y, pred_baseline),
    score(y, pred_updated),
], index=["baseline (904)", "updated (1004)"])

print("\nBoth RF models has the same validation set")
print(results.round(2))
print("\nchange after adding the 100 samples:")
print((results.loc["updated (1004)"] - results.loc["baseline (904)"]).round(2))

#### Spatial Cross-Validation

In [ ]:
coords = np.c_[data.geometry.x.values, data.geometry.y.values]

from scipy.spatial import cKDTree

tree = cKDTree(coords)
nn_dist = tree.query(coords, k=2)[0][:, 1] / 1000.0 # km to nearest neighbour

print("nearest-neighbour distance, km")
print(f"  all 1004 : median {np.median(nn_dist):6.2f}   "
      f"q1 {np.percentile(nn_dist,25):5.2f}  q3 {np.percentile(nn_dist,75):6.2f}")
print(f"  legacy   : median {np.median(nn_dist[~is_new]):6.2f}")
print(f"  new      : median {np.median(nn_dist[is_new]):6.2f}")
print("\nIf the new samples sit much closer together than the legacy ones, the spatial folds will treat them very differently from the random folds.")

#### Residual variogram

In [ ]:
resid = y - pred_baseline
 
edges = np.array([0, 2, 5, 10, 15, 20, 30, 40, 60, 80, 120]) * 1000.0
i, j = np.triu_indices(len(coords), k = 1)
d = np.hypot(coords[i, 0] - coords[j, 0], coords[i, 1] - coords[j, 1])
sq = 0.5 * (resid[i] - resid[j]) ** 2
 
legacy_pair = (~is_new[i]) & (~is_new[j])  # both members of the pair are legacy
sill = resid[~is_new].var()
 
print("semivariance of the baseline residuals, legacy observations only")
print(f"{'bin (km)':>12}{'pairs':>10}{'semivariance':>14}{'/ sill':>9}")
for a, b in zip(edges[:-1], edges[1:]):
    m = (d >= a) & (d < b) & legacy_pair
    if m.sum() > 30:
        print(f"{a/1000:5.0f}-{b/1000:<6.0f}{m.sum():>10}{sq[m].mean():>14.2f}"
              f"{sq[m].mean()/sill:>9.2f}")
print(f"{'sill (variance)':>12}{'':>10}{sill:>14.2f}{1.00:>9.2f}")

#### Spatial block creation based on the residual variogram distance

In [ ]:
# Spatial blocks
block_km = 15 # set this from the variogram above

def make_block_folds(coords, k, seed, block_km):
    """Square blocks of block_km, shuffled and dealt round-robin into k folds."""
    e = block_km * 1000.0 # convert km to meters
    bx = np.floor((coords[:, 0] - coords[:, 0].min()) / e).astype(int)
    by = np.floor((coords[:, 1] - coords[:, 1].min()) / e).astype(int)
    block_id = bx * (by.max() + 1) + by

    unique_blocks = np.unique(block_id)
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(unique_blocks)
    block_to_fold = {b: i % k for i, b in enumerate(shuffled)}
    
    print(f"\n{len(unique_blocks)} blocks of {block_km} km dealt into {k} folds")
    return np.array([block_to_fold[b] for b in block_id])
    
fold_spatial = make_block_folds(coords, n_folds, seed, block_km)

print("samples per fold    :", np.bincount(fold_spatial).tolist())
print("new samples per fold:", [int(is_new[fold_spatial == i].sum()) for i in range(n_folds)])

pred_base_sp = np.full(len(data), np.nan)
pred_upd_sp = np.full(len(data), np.nan)

for i in range(n_folds):
    is_test = (fold_spatial == i)
    is_train = ~is_test
    is_train_baseline = is_train & (~is_new)

    m_u = RandomForestRegressor(**best_params, random_state = seed)
    m_u.fit(X[is_train], y[is_train])
    pred_upd_sp[is_test] = m_u.predict(X[is_test])

    m_b = RandomForestRegressor(**best_params, random_state = seed)
    m_b.fit(X[is_train_baseline], y[is_train_baseline])
    pred_base_sp[is_test] = m_b.predict(X[is_test])

assert not np.isnan(pred_base_sp).any() and not np.isnan(pred_upd_sp).any()

#### Report performance metrics

In [ ]:
print("Spatial block folds")
spatial_results = pd.DataFrame([score(y, pred_base_sp), score(y, pred_upd_sp)],
                               index=["baseline (no new samples)",
                                      "updated (with new samples)"])
print(spatial_results.round(2))
print("\nchange after adding the 100 samples:")
print((spatial_results.loc["updated (with new samples)"]
       - spatial_results.loc["baseline (no new samples)"]).round(2))
